In [2]:
# Create initial guess for M_org1 
import sys
sys.path.append("../code")

import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from inv_ccn_utils import * #includes all CCN closure code from Rahul (execute test run, etc. as functions)
from datetime import datetime


In [6]:
# Import observations
obs_dir = '../input_data'                    

nsd_abs1 = pd.read_csv(os.path.join(obs_dir, 'NSD_mode1.csv'), parse_dates=['datetime'])  # Load NSD data for mode 1
nsd_abs2 = pd.read_csv(os.path.join(obs_dir, 'NSD_mode2.csv'), parse_dates=['datetime'])  # Load NSD data for mode 2
comp_obs = pd.read_csv(os.path.join(obs_dir, 'comp.csv'), parse_dates=['datetime'])  # Load observed composition data, mass fractions/concentrations
ccn_obs = pd.read_csv(os.path.join(obs_dir, 'CCN.csv'), parse_dates=['datetime'])  # Load observed CCN data
dp_dry = np.loadtxt(os.path.join(obs_dir, 'Dp.txt'))  # Dry particle diameters in nanometers
bimodal_params = pd.read_csv(os.path.join(obs_dir, 'Bimodal_parameters.csv'), parse_dates=['datetime'])  # fitted Bimodal parameters 
bimodal_mad = pd.read_csv(os.path.join(obs_dir, 'MAD.csv'), parse_dates=['datetime'])  # Bimodal median absolute deviation
CCN_all = pd.read_csv(os.path.join(obs_dir, 'CCN_all.csv'), parse_dates=['datetime','start_time','end_time'])  # Load all CCN obs
NSD_params_all = pd.read_csv(os.path.join(obs_dir, 'NSD_PARAMS_SCALED.csv'), parse_dates=['datetime'])  # Load all NSD parameters
ACSM = pd.read_csv(os.path.join(obs_dir, 'ACSM_eBC_with_inorganics.csv'), parse_dates=['datetime'])  # ACSM data

if not all(len(df) == len(nsd_abs1) for df in [nsd_abs2, comp_obs, ccn_obs]):
    raise ValueError('Dataframes have different lengths')

Extra = make_EXTRA(dp_dry) # Extra is a dictionary, these initial entries don't change with time.

In [7]:
# look at distributions of M_org1 for idea about how to set up the prior.
# calculate M_org1 depending on fitted NSDs.

M_org1_initial = []
#loop through all timesteps and output the M_org1 values:
for i in range(len(nsd_abs1)):

# Set up ------------------
    # first pre-calculate some parameters:
    NSD1_vec = np.array(nsd_abs1.iloc[i,1:].values) # NSD for mode 1
    NSD2_vec = np.array(nsd_abs2.iloc[i,1:].values) # NSD for mode 2

    response = np.array(ccn_obs.iloc[i,1:].values) # CCN observations at 5 supersaturations

    mass_frac = [
        comp_obs['Org'][i],         # Mass fractions of the components in the particles: [Org, Other, NH4SO4, NH4NO3, BC].
        comp_obs['total_mass'][i],
        comp_obs['NH4SO4'][i],
        comp_obs['NH4NO3'][i],
        comp_obs['eBC880'][i]
        ]

    # mass vectors:
    mass_vec_NH4SO4 = comp_obs['NH4SO4'][i] * comp_obs['total_mass'][i]
    mass_vec_NH4NO3 = comp_obs['NH4NO3'][i] * comp_obs['total_mass'][i]

    # mass fractions
    mass_frac_vec_NH4SO4 = mass_vec_NH4SO4 / (mass_vec_NH4SO4 + mass_vec_NH4NO3)
    mass_frac_vec_NH4NO3 = mass_vec_NH4NO3 / (mass_vec_NH4SO4 + mass_vec_NH4NO3)

    # densities:
    rho_sulp = Extra['densities'][1]   # in kg/m^3
    rho_nitr = Extra['densities'][2]

    # get inorganic density (we include both NH4SO4 and NH4NO3)
    rho_inorg = (mass_frac_vec_NH4SO4 * rho_sulp) + (mass_frac_vec_NH4NO3 * rho_nitr)

    # add to Extra the variables that are calculated each time step:
    Extra['true_inputs'] = mass_frac
    Extra['rho_inorg'] = rho_inorg

    # calculate the mass of the particles in both modes:
    info_mass = cal_mass(dp_dry, Extra['true_inputs'], Extra, NSD1_vec, NSD2_vec)

    M_org1_initial.append(info_mass['M_org1']) # save the M_org1 value for this timestep

In [ ]:
M_org1_initial.to_csv('input_data/M_org1_initialguess.csv', index=False)  # Save the initial guess for M_org1